In [71]:
# Data handling and manipulation library
import pandas as pd

# Library for numerical operations in Python
import numpy as np

# Visualizations library
import matplotlib.pyplot as plt

# Exploratory visualizations library
import seaborn as sns

# Statistical functions library
from scipy import stats

import os

from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


In [72]:
# Now we will load all the CSV files

# Defining where the data is located
data_folder = "../files"

#Creating an empty dictionary to store your DataFrames
dataframes = {}

#Looping through all files in the folder
for filename in os.listdir(data_folder):
    if filename.endswith(".csv"):
        # Build the full file path (e.g. "../files/data1.csv")
        file_path = os.path.join(data_folder, filename)
        # Load the CSV file into a DataFrame
        df = pd.read_csv(file_path, sep=";", encoding="latin-1")
        # Add this DataFrame to our dictionary + remove the .csv from the filename for the key
        name = filename.replace(".csv", "")
        dataframes[name] = df
        print(f"Loaded {filename} -> shape: {df.shape}")

# List of the datasets that have been loaded
list(dataframes.keys())



Loaded Absence.csv -> shape: (60, 3)
Loaded T2D.csv -> shape: (60, 5)
Loaded Longterm_Disease.csv -> shape: (60, 3)
Loaded Fraction_Blood_Clot.csv -> shape: (60, 3)
Loaded WHO_Minimum_Physical_Activity.csv -> shape: (60, 3)
Loaded Morbidly_Obese.csv -> shape: (60, 3)
Loaded Sales-Bricks.csv -> shape: (3389, 5)
Loaded Patient_Data.csv -> shape: (428, 1)
Loaded Life_Expectancy.csv -> shape: (60, 3)
Loaded Education_Long.csv -> shape: (60, 6)
Loaded Attention.csv -> shape: (21113, 7)


['Absence',
 'T2D',
 'Longterm_Disease',
 'Fraction_Blood_Clot',
 'WHO_Minimum_Physical_Activity',
 'Morbidly_Obese',
 'Sales-Bricks',
 'Patient_Data',
 'Life_Expectancy',
 'Education_Long',
 'Attention']

In [73]:
# === 3) Take a quick look at the loaded datasets ===

# Cell 3 - Quick per-file inspection
for name, df in dataframes.items():
    print("===")
    print(name)
    print("shape:", df.shape)
    print("columns:", list(df.columns[:10]) + (["..."] if len(df.columns)>10 else []))
    print("dtypes (top):")
    print(df.dtypes[:10])
    print("missing (top 8):")
    print(df.isna().sum().sort_values(ascending=False).head(8))
    print()


===
Absence
shape: (60, 3)
columns: ['brick_nr', 'brick', 'fravær']
dtypes (top):
brick_nr     int64
brick       object
fravær      object
dtype: object
missing (top 8):
brick_nr    0
brick       0
fravær      0
dtype: int64

===
T2D
shape: (60, 5)
columns: ['brick_nr', 'brick', 'patients', 'population', 'patients_per_1000']
dtypes (top):
brick_nr              int64
brick                object
patients             object
population            int64
patients_per_1000    object
dtype: object
missing (top 8):
brick_nr             0
brick                0
patients             0
population           0
patients_per_1000    0
dtype: int64

===
Longterm_Disease
shape: (60, 3)
columns: ['brick_nr', 'brick', 'anddel']
dtypes (top):
brick_nr     int64
brick       object
anddel      object
dtype: object
missing (top 8):
brick_nr    0
brick       0
anddel      0
dtype: int64

===
Fraction_Blood_Clot
shape: (60, 3)
columns: ['brick_nr', 'brick', 'Andel med blodprop i hjertet']
dtypes (top):
brick_nr

In [74]:
# Now we will clean and standardize the columns of all the datasets so we can merge them afterwards

# We'll loop through all the DataFrames again
for name, df in dataframes.items():
    # Create a cleaned version of each column name
    cleaned_columns = []
    for col in df.columns:
        new_col = (
            col.strip()            # remove leading/trailing spaces
               .lower()            # convert to lowercase
               .replace(" ", "_")  # replace spaces with underscores
               .replace("-", "_")  # replace dashes with underscores
        )
        # You can also remove parentheses or special symbols if needed
        for ch in "()[]{}$%#@!":
            new_col = new_col.replace(ch, "")
        cleaned_columns.append(new_col)

    # Apply the cleaned column names to the DataFrame
    df.columns = cleaned_columns

    # Update the DataFrame in the dictionary (not strictly necessary, but explicit)
    dataframes[name] = df

# Show the cleaned column names for each dataset
for name, df in dataframes.items():
    print(f"{name}: {list(df.columns)}")


Absence: ['brick_nr', 'brick', 'fravær']
T2D: ['brick_nr', 'brick', 'patients', 'population', 'patients_per_1000']
Longterm_Disease: ['brick_nr', 'brick', 'anddel']
Fraction_Blood_Clot: ['brick_nr', 'brick', 'andel_med_blodprop_i_hjertet']
WHO_Minimum_Physical_Activity: ['brick_nr', 'brick', 'andel']
Morbidly_Obese: ['brick_nr', 'brick', 'andel']
Sales-Bricks: ['year_month', 'brick_code', 'volume', 'value', 'market']
Patient_Data: ['hospital,department,n_type1,n_type2,,,,,,,,,']
Life_Expectancy: ['brick_nr', 'brick', 'år']
Education_Long: ['brick_nr', 'brick', 'antal', 'brick_navn', 'population', 'value_per_1000']
Attention: ['ï»¿site_short', 'period_id', 'count_', 'who_atc_code', 'category', 'brick_name', 'brick_no']


In [75]:
# === 5) Basic data cleaning for each dataset ===

for name, df in dataframes.items():
    print("====================================================")
    print(f"🧽 Cleaning dataset: {name}")
    print(f"Original shape: {df.shape}")

    # --- 1. Remove duplicate rows ---
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"➡️ Removed {before - after} duplicate rows")

    # --- 2. Handle missing values ---
    # You can choose how to handle them depending on the situation.
    # For now, let's just show how many missing values we have:
    missing = df.isna().sum().sum()
    print(f"🔍 Total missing values: {missing}")

    # Option A (safe start): Keep all data and deal with missing later
    # Option B (aggressive): Drop rows with any missing values
    # df = df.dropna()

    # --- 3. Fix data types ---
    # Example: convert columns that look like dates to datetime objects
    for col in df.columns:
        if "date" in col or "time" in col:
            try:
                df[col] = pd.to_datetime(df[col])
                print(f"📅 Converted '{col}' to datetime")
            except Exception:
                pass  # skip columns that can't be converted

    # Example: convert numeric-looking text columns to numbers
    for col in df.columns:
        if df[col].dtype == "object":
            try:
                df[col] = pd.to_numeric(df[col])
                print(f"🔢 Converted '{col}' to numeric")
            except Exception:
                pass  # skip if it can't be converted

    # --- 4. Trim whitespace from text columns ---
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()

    # Save the cleaned version back to the dictionary
    dataframes[name] = df

    print(f"✅ Finished cleaning {name} — new shape: {df.shape}")
    print("====================================================\n")



🧽 Cleaning dataset: Absence
Original shape: (60, 3)
➡️ Removed 0 duplicate rows
🔍 Total missing values: 0
✅ Finished cleaning Absence — new shape: (60, 3)

🧽 Cleaning dataset: T2D
Original shape: (60, 5)
➡️ Removed 0 duplicate rows
🔍 Total missing values: 0
✅ Finished cleaning T2D — new shape: (60, 5)

🧽 Cleaning dataset: Longterm_Disease
Original shape: (60, 3)
➡️ Removed 0 duplicate rows
🔍 Total missing values: 0
✅ Finished cleaning Longterm_Disease — new shape: (60, 3)

🧽 Cleaning dataset: Fraction_Blood_Clot
Original shape: (60, 3)
➡️ Removed 0 duplicate rows
🔍 Total missing values: 0
✅ Finished cleaning Fraction_Blood_Clot — new shape: (60, 3)

🧽 Cleaning dataset: WHO_Minimum_Physical_Activity
Original shape: (60, 3)
➡️ Removed 0 duplicate rows
🔍 Total missing values: 0
✅ Finished cleaning WHO_Minimum_Physical_Activity — new shape: (60, 3)

🧽 Cleaning dataset: Morbidly_Obese
Original shape: (60, 3)
➡️ Removed 0 duplicate rows
🔍 Total missing values: 0
✅ Finished cleaning Morbidly_

In [76]:
# Save all cleaned dataframes to a new folder
output_folder = "../cleaned_files"
os.makedirs(output_folder, exist_ok=True)

for name, df in dataframes.items():
    path = os.path.join(output_folder, f"{name}_cleaned.csv")
    df.to_csv(path, index=False)

In [77]:
# === 6) Exploratory Data Analysis (EDA) ===

# See column names for all datasets
for name, df in dataframes.items():
    print(f"{name}: {list(df.columns)}")


Absence: ['brick_nr', 'brick', 'fravær']
T2D: ['brick_nr', 'brick', 'patients', 'population', 'patients_per_1000']
Longterm_Disease: ['brick_nr', 'brick', 'anddel']
Fraction_Blood_Clot: ['brick_nr', 'brick', 'andel_med_blodprop_i_hjertet']
WHO_Minimum_Physical_Activity: ['brick_nr', 'brick', 'andel']
Morbidly_Obese: ['brick_nr', 'brick', 'andel']
Sales-Bricks: ['year_month', 'brick_code', 'volume', 'value', 'market']
Patient_Data: ['hospital,department,n_type1,n_type2,,,,,,,,,']
Life_Expectancy: ['brick_nr', 'brick', 'år']
Education_Long: ['brick_nr', 'brick', 'antal', 'brick_navn', 'population', 'value_per_1000']
Attention: ['ï»¿site_short', 'period_id', 'count_', 'who_atc_code', 'category', 'brick_name', 'brick_no']
